# 00 | Trích đặc trưng V4 từ tín hiệu thô MIMIC PERform AF

**Mục đích.** Bước đầu tiên của pipeline V4: đọc PPG thô của 35 bệnh nhân, lọc, dò nhịp, tính chuỗi NN, cắt cửa sổ 30 giây bước 10 giây và tính 16 đặc trưng HRV cho từng cửa sổ, kèm `record_id` để mọi bước sau chia dữ liệu theo bệnh nhân. Kết quả là bảng đặc trưng mà notebook 01 và notebook `review/evaluate_v4` dùng.

| | |
|---|---|
| Đầu vào | `data/raw/mimic_perform/` (`af/`, `non-af/`); nếu thiếu, ô 2 tự tải bản công khai |
| Đầu ra | `data/features/v4/mimic_features_v4.csv` |
| Thư viện | `healthsense_core.feature_extraction.extract_all`, bên trong gọi `signal_processing` và `hrv_features` |
| Thời gian | khoảng 1 đến 3 phút |

Tuỳ chọn ở cuối: trích thêm đặc trưng AFDB (ECG) cho cross-dataset và pooled model, cần tải annotation từ PhysioNet.

In [1]:
# Tìm gốc repo và nạp thư viện src/healthsense_core (chạy được từ bất kỳ thư mục con nào)
import os, sys, json, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "healthsense_core").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Không tìm thấy gốc repo (thư mục chứa src/healthsense_core)")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Repo root:", ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

Repo root: D:\Projects\HealthSense\HealthSense-MachineLearning-Lab


In [2]:
from healthsense_core import config
from healthsense_core.feature_extraction import extract_all, load_features

print(f"Cửa sổ {config.WINDOW_S} s, bước {config.STEP_S} s, tối thiểu {config.MIN_BEATS_PER_WINDOW} nhịp/cửa sổ")
print("Tín hiệu thô:", os.path.relpath(config.RAW_DIR))
if not os.path.isdir(config.RAW_DIR):
    from healthsense_core.data_loading import download_dataset
    print("Chưa có dữ liệu thô, đang tải bản công khai (khoảng 100 MB)...")
    download_dataset()

Cửa sổ 30 s, bước 10 s, tối thiểu 10 nhịp/cửa sổ
Tín hiệu thô: data\raw\mimic_perform


## 1. Trích đặc trưng cho toàn bộ bệnh nhân

Mỗi bản ghi đi qua đúng pipeline mô tả ở mục IV-B và IV-C của bài báo. Cửa sổ không đủ số nhịp bị bỏ.

In [3]:
t0 = time.time()
extract_all(verbose=True)
print(f"Xong trong {time.time()-t0:.0f}s")

  [ 1/35] mimic_perform_af_001 (AFib): 118 cửa sổ


  [ 2/35] mimic_perform_af_002 (AFib): 118 cửa sổ


  [ 3/35] mimic_perform_af_003 (AFib): 118 cửa sổ


  [ 4/35] mimic_perform_af_004 (AFib): 118 cửa sổ


  [ 5/35] mimic_perform_af_005 (AFib): 118 cửa sổ


...

  [31/35] mimic_perform_non_af_012 (Normal): 118 cửa sổ


  [32/35] mimic_perform_non_af_013 (Normal): 118 cửa sổ


  [33/35] mimic_perform_non_af_014 (Normal): 118 cửa sổ


  [34/35] mimic_perform_non_af_015 (Normal): 118 cửa sổ


  [35/35] mimic_perform_non_af_016 (Normal): 118 cửa sổ

Tổng: 4130 cửa sổ (2242 AFib / 1888 Normal) từ 35 bệnh nhân
Đã lưu: D:\Projects\HealthSense\HealthSense-MachineLearning-Lab\data\features\v4\mimic_features_v4.csv
Xong trong 14s


## 2. Kiểm tra bảng đặc trưng vừa sinh

In [4]:
df = load_features()
print(f"{len(df)} cửa sổ | {df.record_id.nunique()} bệnh nhân | AF {df.status.mean():.1%}")
print("Đã lưu:", os.path.relpath(config.FEATURES_V4_FILE))
df.groupby("status").agg(cửa_sổ=("record_id", "size"), bệnh_nhân=("record_id", "nunique"))

4130 cửa sổ | 35 bệnh nhân | AF 54.3%
Đã lưu: data\features\v4\mimic_features_v4.csv


,cửa_sổ,bệnh_nhân
status,,
0,1888,16
1,2242,19


In [5]:
df[config.ALL_FEATURES].describe().T[["mean", "std", "min", "max"]].round(3)

,mean,std,min,max
HR_mean,87.706,18.692,43.940,132.035
Mean_NN,718.802,168.613,454.424,1365.500
SDNN,127.043,96.026,4.316,564.133
RMSSD,168.541,126.942,5.950,785.779
NN50,22.011,14.873,0.000,52.000
pNN50,52.014,32.661,0.000,100.000
CV,0.182,0.128,0.008,0.617
LF,7609.454,71765.092,0.000,3526786.312
HF,11118.117,27046.153,0.721,528586.874
Total_Power,28471.406,217192.068,2.289,9737581.200


## 3. Tuỳ chọn: đặc trưng AFDB (ECG) cho cross-dataset

Pooled model V4 và kiểm định chéo cần bảng đặc trưng của MIT-BIH AFDB. Hàm `afdb.load_features()` tự tải annotation từ PhysioNet và trích nếu chưa có file. Đặt `EXTRACT_AFDB = True` khi cần; mặc định bỏ qua nếu file đã tồn tại.

In [6]:
EXTRACT_AFDB = False
from healthsense_core import afdb
if EXTRACT_AFDB or not os.path.exists(afdb.FEATURES_AFDB_FILE):
    af = afdb.load_features()
    print(f"AFDB: {len(af)} cửa sổ, {af.record_id.nunique()} bản ghi -> {os.path.relpath(afdb.FEATURES_AFDB_FILE)}")
else:
    print("Đã có:", os.path.relpath(afdb.FEATURES_AFDB_FILE))

Đã có: data\features\v5\afdb_features_v5_afib_only.csv
